# Modelo de Regresión Lineal basado en productos "mágicos"

In [15]:
# EJECUTAR LIBRERIAS
import os
import pandas as pd

In [16]:
# LEEMOS LOS DATOS
drive_base_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Data/'
filename = 'sell-in.txt'
filepath = os.path.join(drive_base_path, filename)
sell = pd.read_csv(filepath, sep='\t')
print(sell.head(10))

filename = 'product_id_apredecir201912.txt'
filepath = os.path.join(drive_base_path, filename)
a_predecir = pd.read_csv(filepath, sep='\t')

   periodo  customer_id  product_id  plan_precios_cuidados  cust_request_qty  \
0   201701        10234       20524                      0                 2   
1   201701        10032       20524                      0                 1   
2   201701        10217       20524                      0                 1   
3   201701        10125       20524                      0                 1   
4   201701        10012       20524                      0                11   
5   201701        10080       20524                      0                 1   
6   201701        10015       20524                      0                 4   
7   201701        10062       20524                      0                 1   
8   201701        10159       20524                      0                 3   
9   201701        10183       20524                      0                 1   

   cust_request_tn       tn  
0          0.05300  0.05300  
1          0.13628  0.13628  
2          0.03028  0.03028  

In [17]:
# Veo la cantidad de valores distintos en "periodo"
num_periodos = sell['periodo'].nunique()
print(f'Cantidad de valores distintos en "periodo": {num_periodos}')

Cantidad de valores distintos en "periodo": 36


In [18]:
# Agrupar y sumar
sell_agrup = (
    sell
    .groupby(['periodo', 'product_id'], as_index=False)['tn']
    .sum()
)

print(sell_agrup)

       periodo  product_id          tn
0       201701       20001   934.77222
1       201701       20002   550.15707
2       201701       20003  1063.45835
3       201701       20004   555.91614
4       201701       20005   494.27011
...        ...         ...         ...
31238   201912       21265     0.05007
31239   201912       21266     0.05121
31240   201912       21267     0.01569
31241   201912       21271     0.00298
31242   201912       21276     0.00892

[31243 rows x 3 columns]


In [19]:
# FEATURE ENGINEERING Y CREACION DE LA CLASE A PREDECIR
# Ordenar por product_id y periodo para asegurar el orden correcto
sell_agrup = sell_agrup.sort_values(['product_id', 'periodo']).reset_index(drop=True)

# Crear las 11 columnas con los valores de los períodos anteriores
for i in range(1, 12):  # Del 1 al 11
    sell_agrup[f'tn_lag_{i}'] = sell_agrup.groupby('product_id')['tn'].shift(i)

# Agregar columna con el valor de tn del período +2 (2 períodos hacia adelante)
sell_agrup['tn_target'] = sell_agrup.groupby('product_id')['tn'].shift(-2)

# Mostrar el resultado
print("Dataset con las 11 columnas de períodos anteriores y 1 columna de período futuro:")
print(sell_agrup.head(36))
print(f"\nForma del dataset: {sell_agrup.shape}")
print(f"Columnas: {list(sell_agrup.columns)}")

Dataset con las 11 columnas de períodos anteriores y 1 columna de período futuro:
    periodo  product_id          tn    tn_lag_1    tn_lag_2    tn_lag_3  \
0    201701       20001   934.77222         NaN         NaN         NaN   
1    201702       20001   798.01620   934.77222         NaN         NaN   
2    201703       20001  1303.35771   798.01620   934.77222         NaN   
3    201704       20001  1069.96130  1303.35771   798.01620   934.77222   
4    201705       20001  1502.20132  1069.96130  1303.35771   798.01620   
5    201706       20001  1520.06539  1502.20132  1069.96130  1303.35771   
6    201707       20001  1030.67391  1520.06539  1502.20132  1069.96130   
7    201708       20001  1267.39462  1030.67391  1520.06539  1502.20132   
8    201709       20001  1316.94604  1267.39462  1030.67391  1520.06539   
9    201710       20001  1439.75563  1316.94604  1267.39462  1030.67391   
10   201711       20001  1580.47401  1439.75563  1316.94604  1267.39462   
11   201712       

In [20]:
# Definir la lista de product_id específicos
magicos = [20002, 20003, 20006, 20010, 20011, 20018, 20019, 20021,
           20026, 20028, 20035, 20039, 20042, 20044, 20045, 20046, 20049,
           20051, 20052, 20053, 20055, 20008, 20001, 20017, 20086, 20180,
           20193, 20320, 20532, 20612, 20637, 20807, 20838]

# Crear el subconjunto filtrado por período 201812 y los product_id específicos
sell_agrup_subset = sell_agrup[
    (sell_agrup['periodo'] == 201812) & 
    (sell_agrup['product_id'].isin(magicos))
]

print(f"Dataset original shape: {sell_agrup.shape}")
print(f"Subconjunto filtrado shape: {sell_agrup_subset.shape}")
print(f"Cantidad de productos únicos en el subconjunto: {sell_agrup_subset['product_id'].nunique()}")
print("\nPrimeras filas del subconjunto:")
print(sell_agrup_subset.head())

Dataset original shape: (31243, 15)
Subconjunto filtrado shape: (33, 15)
Cantidad de productos únicos en el subconjunto: 33

Primeras filas del subconjunto:
     periodo  product_id          tn    tn_lag_1    tn_lag_2    tn_lag_3  \
23    201812       20001  1486.68669  1813.01511  2295.19832  1438.67455   
59    201812       20002  1009.45458  1766.81068  1378.49032   954.23575   
95    201812       20003   769.82869  1206.91773  1313.34211   912.34156   
203   201812       20006   407.75925   566.66809   513.15472   478.04388   
275   201812       20008   426.32899   433.50170   532.45644   436.96269   

       tn_lag_4    tn_lag_5    tn_lag_6    tn_lag_7    tn_lag_8    tn_lag_9  \
23   1800.96168  1470.41009  1150.79169  1293.89788  1251.28462  1856.83534   
59   1161.88430   977.40239  1033.82845  1103.39191   999.20934   966.86044   
95    955.97079   656.22700   660.73323   784.35885   765.47838   778.55594   
203   615.70617   515.20419   468.15260   865.28861   748.44391   862.

In [21]:
# Importar librerías necesarias para regresión lineal
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# Preparar los datos para el modelo
# Columnas 3 a 14 como predictores (tn_lag_1 hasta tn_lag_11 y tn)
X = sell_agrup_subset.iloc[:, 2:14]  # Columnas 3 a 14 (índices 2 a 13)
# Columna 15 como variable target
y = sell_agrup_subset.iloc[:, 14]    # Columna 15 (índice 14)

print("Variables predictoras (X):")
print(f"Shape: {X.shape}")
print(f"Columnas: {list(X.columns)}")
print("\nVariable target (y):")
print(f"Shape: {y.shape}")
print(f"Nombre: {y.name}")

# Verificar si hay valores NaN y eliminar filas con NaN
print(f"\nValores NaN en X: {X.isnull().sum().sum()}")
print(f"Valores NaN en y: {y.isnull().sum()}")

# Eliminar filas con valores NaN
mask = ~(X.isnull().any(axis=1) | y.isnull())
X_clean = X[mask]
y_clean = y[mask]

print(f"\nDatos después de eliminar NaN:")
print(f"X_clean shape: {X_clean.shape}")
print(f"y_clean shape: {y_clean.shape}")

# Ajustar el modelo de regresión lineal
if len(X_clean) > 0:
    modelo = LinearRegression()
    modelo.fit(X_clean, y_clean)
    
    # Hacer predicciones
    y_pred = modelo.predict(X_clean)
    
    # Calcular métricas
    r2 = r2_score(y_clean, y_pred)
    mse = mean_squared_error(y_clean, y_pred)
    mae = mean_absolute_error(y_clean, y_pred)
    rmse = np.sqrt(mse)
    
    print(f"\n=== RESULTADOS DEL MODELO ===")
    print(f"R² Score: {r2:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    
    print(f"\n=== COEFICIENTES ===")
    print(f"Intercepto: {modelo.intercept_:.4f}")
    for i, coef in enumerate(modelo.coef_):
        print(f"{X_clean.columns[i]}: {coef:.4f}")
else:
    print("No hay suficientes datos sin NaN para ajustar el modelo.")

Variables predictoras (X):
Shape: (33, 12)
Columnas: ['tn', 'tn_lag_1', 'tn_lag_2', 'tn_lag_3', 'tn_lag_4', 'tn_lag_5', 'tn_lag_6', 'tn_lag_7', 'tn_lag_8', 'tn_lag_9', 'tn_lag_10', 'tn_lag_11']

Variable target (y):
Shape: (33,)
Nombre: tn_target

Valores NaN en X: 0
Valores NaN en y: 0

Datos después de eliminar NaN:
X_clean shape: (33, 12)
y_clean shape: (33,)

=== RESULTADOS DEL MODELO ===
R² Score: 0.9883
MSE: 954.9393
RMSE: 30.9021
MAE: 21.0201

=== COEFICIENTES ===
Intercepto: 0.4415
tn: -0.0013
tn_lag_1: 0.2366
tn_lag_2: 0.1782
tn_lag_3: -0.0600
tn_lag_4: -0.1619
tn_lag_5: -0.0078
tn_lag_6: 0.1519
tn_lag_7: 0.0439
tn_lag_8: 0.1428
tn_lag_9: 0.1038
tn_lag_10: 0.1192
tn_lag_11: 0.0737


In [22]:
# Filtrar sell_agrup por los product_id presentes en a_predecir
sell_agrup_filtrado = sell_agrup[sell_agrup['product_id'].isin(a_predecir['product_id'])]
print(sell_agrup_filtrado)

drive_base_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression'
filename = 'sell_agrup_filtrado.txt'
filepath = os.path.join(drive_base_path, filename)

# Guardar el DataFrame filtrado en un archivo de texto
sell_agrup_filtrado.to_csv(filepath, sep='\t', index=False)

       periodo  product_id          tn    tn_lag_1    tn_lag_2   tn_lag_3  \
0       201701       20001   934.77222         NaN         NaN        NaN   
1       201702       20001   798.01620   934.77222         NaN        NaN   
2       201703       20001  1303.35771   798.01620   934.77222        NaN   
3       201704       20001  1069.96130  1303.35771   798.01620  934.77222   
4       201705       20001  1502.20132  1069.96130  1303.35771  798.01620   
...        ...         ...         ...         ...         ...        ...   
31206   201908       21276     0.01265     0.00223     0.04086    0.09283   
31207   201909       21276     0.01856     0.01265     0.00223    0.04086   
31208   201910       21276     0.02079     0.01856     0.01265    0.00223   
31209   201911       21276     0.03341     0.02079     0.01856    0.01265   
31210   201912       21276     0.00892     0.03341     0.02079    0.01856   

        tn_lag_4  tn_lag_5  tn_lag_6  tn_lag_7  tn_lag_8  tn_lag_9  tn_lag_

In [23]:
# Filtrar los períodos de interés
periodos_interes = [201901, 201902, 201903, 201904, 201905, 201906,
                    201907, 201908, 201909, 201910, 201911, 201912]
sell_agrup_filtrado_pi = sell_agrup_filtrado[sell_agrup_filtrado['periodo'].isin(periodos_interes)]

In [24]:
# Hacer predicciones para todos los product_id en sell_agrup_filtrado_pi
print("=== APLICANDO PREDICCIONES ===")

# Obtener los coeficientes del modelo para usar manualmente
if 'modelo' in locals():
    coef_tn = modelo.coef_[0]  # Coeficiente para 'tn'
    coef_lags = modelo.coef_[1:]  # Coeficientes para tn_lag_1 hasta tn_lag_11
    intercepto = modelo.intercept_
    
    print(f"Coeficiente tn: {coef_tn:.4f}")
    print(f"Coeficientes lag: {coef_lags}")
    print(f"Intercepto: {intercepto:.4f}")
    
    # Crear dataset para predicciones
    predicciones = []
    
    # Obtener todos los product_id únicos
    productos_unicos = sell_agrup_filtrado_pi['product_id'].unique()
    
    for product_id in productos_unicos:
        # Filtrar datos del producto específico
        datos_producto = sell_agrup_filtrado_pi[sell_agrup_filtrado_pi['product_id'] == product_id].copy()
        datos_producto = datos_producto.sort_values('periodo')
        
        # Verificar si tiene datos para todos los 12 períodos (201901 a 201912)
        periodos_completos = [201901, 201902, 201903, 201904, 201905, 201906,
                             201907, 201908, 201909, 201910, 201911, 201912]
        
        periodos_disponibles = set(datos_producto['periodo'].tolist())
        tiene_todos_periodos = all(p in periodos_disponibles for p in periodos_completos)
        
        if tiene_todos_periodos:
            # CASO 1: Usar el modelo de regresión lineal
            # Obtener valores específicos para la predicción manual
            tn_201912 = datos_producto[datos_producto['periodo'] == 201912]['tn'].iloc[0]
            tn_201911 = datos_producto[datos_producto['periodo'] == 201911]['tn'].iloc[0]
            tn_201910 = datos_producto[datos_producto['periodo'] == 201910]['tn'].iloc[0]
            tn_201909 = datos_producto[datos_producto['periodo'] == 201909]['tn'].iloc[0]
            tn_201908 = datos_producto[datos_producto['periodo'] == 201908]['tn'].iloc[0]
            tn_201907 = datos_producto[datos_producto['periodo'] == 201907]['tn'].iloc[0]
            tn_201906 = datos_producto[datos_producto['periodo'] == 201906]['tn'].iloc[0]
            tn_201905 = datos_producto[datos_producto['periodo'] == 201905]['tn'].iloc[0]
            tn_201904 = datos_producto[datos_producto['periodo'] == 201904]['tn'].iloc[0]
            tn_201903 = datos_producto[datos_producto['periodo'] == 201903]['tn'].iloc[0]
            tn_201902 = datos_producto[datos_producto['periodo'] == 201902]['tn'].iloc[0]
            tn_201901 = datos_producto[datos_producto['periodo'] == 201901]['tn'].iloc[0]
            
            # Aplicar la fórmula manual del modelo
            prediccion = (intercepto + 
                         coef_tn * tn_201912 +           # tn actual
                         coef_lags[0] * tn_201911 +      # tn_lag_1
                         coef_lags[1] * tn_201910 +      # tn_lag_2
                         coef_lags[2] * tn_201909 +      # tn_lag_3
                         coef_lags[3] * tn_201908 +      # tn_lag_4
                         coef_lags[4] * tn_201907 +      # tn_lag_5
                         coef_lags[5] * tn_201906 +      # tn_lag_6
                         coef_lags[6] * tn_201905 +      # tn_lag_7
                         coef_lags[7] * tn_201904 +      # tn_lag_8
                         coef_lags[8] * tn_201903 +      # tn_lag_9
                         coef_lags[9] * tn_201902 +      # tn_lag_10
                         coef_lags[10] * tn_201901)      # tn_lag_11
            
            metodo = "Modelo Regresión"
            
        else:
            # CASO 2: Usar promedio de los períodos disponibles
            promedio = datos_producto['tn'].mean()
            prediccion = promedio
            metodo = "Promedio Períodos"
        
        # Agregar resultado
        predicciones.append({
            'product_id': product_id,
            'prediccion_202002': prediccion,
            'metodo_usado': metodo,
            'periodos_disponibles': len(datos_producto),
            'tiene_12_periodos': tiene_todos_periodos
        })
    
    # Crear DataFrame con las predicciones
    df_predicciones = pd.DataFrame(predicciones)
    
    print(f"\n=== RESUMEN DE PREDICCIONES ===")
    print(f"Total productos procesados: {len(df_predicciones)}")
    print(f"Productos con modelo de regresión: {sum(df_predicciones['metodo_usado'] == 'Modelo Regresión')}")
    print(f"Productos con promedio: {sum(df_predicciones['metodo_usado'] == 'Promedio Períodos')}")
    
    print(f"\n=== PRIMERAS PREDICCIONES ===")
    print(df_predicciones.head(10))
    
    # Mostrar estadísticas de las predicciones
    print(f"\n=== ESTADÍSTICAS DE PREDICCIONES ===")
    print(f"Predicción mínima: {df_predicciones['prediccion_202002'].min():.2f}")
    print(f"Predicción máxima: {df_predicciones['prediccion_202002'].max():.2f}")
    print(f"Predicción promedio: {df_predicciones['prediccion_202002'].mean():.2f}")
    print(f"Desviación estándar: {df_predicciones['prediccion_202002'].std():.2f}")
    
else:
    print("Error: El modelo no está disponible. Ejecuta primero la celda anterior.")

=== APLICANDO PREDICCIONES ===
Coeficiente tn: -0.0013
Coeficientes lag: [ 0.23655828  0.17820788 -0.0600306  -0.16187541 -0.00777452  0.15193647
  0.04393265  0.14283912  0.10380433  0.11921107  0.07367052]
Intercepto: 0.4415

=== RESUMEN DE PREDICCIONES ===
Total productos procesados: 780
Productos con modelo de regresión: 650
Productos con promedio: 130

=== PRIMERAS PREDICCIONES ===
   product_id  prediccion_202002      metodo_usado  periodos_disponibles  \
0       20001        1162.707525  Modelo Regresión                    12   
1       20002        1183.640604  Modelo Regresión                    12   
2       20003         684.763931  Modelo Regresión                    12   
3       20004         580.484961  Modelo Regresión                    12   
4       20005         563.560780  Modelo Regresión                    12   
5       20006         482.886867  Modelo Regresión                    12   
6       20007         390.924420  Modelo Regresión                    12   
7 

In [25]:
# Preparar dataset para exportar con las columnas requeridas
df_export = df_predicciones[['product_id', 'prediccion_202002']].copy()

# Renombrar la columna prediccion_202002 a tn
df_export = df_export.rename(columns={'prediccion_202002': 'tn'})

# Mostrar el dataset que se va a exportar
print("Dataset a exportar:")
print(df_export.head(10))
print(f"\nShape del dataset: {df_export.shape}")
print(f"Columnas: {list(df_export.columns)}")

# Definir la ruta de salida
drive_base_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression'
filename = 'predicciones_202002.csv'
filepath = os.path.join(drive_base_path, filename)

# Crear el directorio si no existe
os.makedirs(drive_base_path, exist_ok=True)

# Guardar el archivo CSV separado por comas
df_export.to_csv(filepath, sep=',', index=False)

print(f"\nArchivo guardado exitosamente en: {filepath}")
print(f"Total de registros guardados: {len(df_export)}")

# Verificar que el archivo se guardó correctamente
if os.path.exists(filepath):
    file_size = os.path.getsize(filepath)
    print(f"Tamaño del archivo: {file_size} bytes")
    
    # Leer las primeras líneas para verificar el formato
    with open(filepath, 'r') as f:
        primeras_lineas = [f.readline().strip() for _ in range(5)]
    
    print(f"\nPrimeras líneas del archivo CSV:")
    for i, linea in enumerate(primeras_lineas):
        print(f"Línea {i+1}: {linea}")
else:
    print("Error: El archivo no se pudo crear.")

Dataset a exportar:
   product_id           tn
0       20001  1162.707525
1       20002  1183.640604
2       20003   684.763931
3       20004   580.484961
4       20005   563.560780
5       20006   482.886867
6       20007   390.924420
7       20008   422.340199
8       20009   450.560618
9       20010   418.689888

Shape del dataset: (780, 2)
Columnas: ['product_id', 'tn']

Archivo guardado exitosamente en: C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression\predicciones_202002.csv
Total de registros guardados: 780
Tamaño del archivo: 19470 bytes

Primeras líneas del archivo CSV:
Línea 1: product_id,tn
Línea 2: 20001,1162.7075252208506
Línea 3: 20002,1183.640604197152
Línea 4: 20003,684.7639305138391
Línea 5: 20004,580.4849611289101


## Predicción con modelo regresión Ridge

In [27]:
# Importar Ridge y herramientas adicionales para comparación de modelos
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

print("=== CONFIGURACIÓN MODELO RIDGE ===")

# Preparar los mismos datos que usamos para regresión lineal
X_ridge = X_clean.copy()
y_ridge = y_clean.copy()

print(f"Datos para Ridge - X shape: {X_ridge.shape}, y shape: {y_ridge.shape}")

# Estandarizar las variables (importante para Ridge)
scaler = StandardScaler()
X_ridge_scaled = scaler.fit_transform(X_ridge)

print("Variables estandarizadas para modelo Ridge")
print(f"Media de X_ridge_scaled: {X_ridge_scaled.mean(axis=0)}")
print(f"Std de X_ridge_scaled: {X_ridge_scaled.std(axis=0)}")

=== CONFIGURACIÓN MODELO RIDGE ===
Datos para Ridge - X shape: (33, 12), y shape: (33,)
Variables estandarizadas para modelo Ridge
Media de X_ridge_scaled: [ 9.42007415e-17 -1.68215610e-17  9.08364293e-17  2.69144976e-17
  1.78308546e-16 -1.44665424e-16 -1.07657990e-16 -4.37360585e-17
  7.73791805e-17 -1.00929366e-17  4.37360585e-17 -1.00929366e-16]
Std de X_ridge_scaled: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [28]:
# Buscar el mejor valor de alpha (regularización) para Ridge
print("=== BÚSQUEDA DEL MEJOR ALPHA PARA RIDGE ===")

# Definir rango de valores alpha para probar
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
best_alpha = None
best_score = -np.inf
alpha_scores = []

# Probar cada valor de alpha con validación cruzada
for alpha in alphas:
    ridge_temp = Ridge(alpha=alpha)
    # Usar validación cruzada (si hay suficientes datos)
    if len(X_ridge_scaled) >= 5:  # Mínimo 5 muestras para CV
        scores = cross_val_score(ridge_temp, X_ridge_scaled, y_ridge, cv=min(5, len(X_ridge_scaled)), scoring='r2')
        mean_score = scores.mean()
    else:
        # Si hay pocos datos, usar el R² del ajuste completo
        ridge_temp.fit(X_ridge_scaled, y_ridge)
        y_pred_temp = ridge_temp.predict(X_ridge_scaled)
        mean_score = r2_score(y_ridge, y_pred_temp)
    
    alpha_scores.append(mean_score)
    print(f"Alpha: {alpha:>6} - R² promedio: {mean_score:.4f}")
    
    if mean_score > best_score:
        best_score = mean_score
        best_alpha = alpha

print(f"\n=== MEJOR CONFIGURACIÓN ===")
print(f"Mejor alpha: {best_alpha}")
print(f"Mejor R² score: {best_score:.4f}")

# Entrenar el modelo Ridge final con el mejor alpha
modelo_ridge = Ridge(alpha=best_alpha)
modelo_ridge.fit(X_ridge_scaled, y_ridge)

=== BÚSQUEDA DEL MEJOR ALPHA PARA RIDGE ===
Alpha:  0.001 - R² promedio: -1.1462
Alpha:   0.01 - R² promedio: -0.7908
Alpha:    0.1 - R² promedio: -0.1331
Alpha:      1 - R² promedio: -0.2774
Alpha:     10 - R² promedio: 0.1094
Alpha:    100 - R² promedio: -39.1687
Alpha:   1000 - R² promedio: -479.1093

=== MEJOR CONFIGURACIÓN ===
Mejor alpha: 10
Mejor R² score: 0.1094


Ridge(alpha=10)

In [29]:
# Evaluar el modelo Ridge y comparar con regresión lineal
print("=== EVALUACIÓN MODELO RIDGE ===")

# Hacer predicciones con Ridge
y_pred_ridge = modelo_ridge.predict(X_ridge_scaled)

# Calcular métricas para Ridge
r2_ridge = r2_score(y_ridge, y_pred_ridge)
mse_ridge = mean_squared_error(y_ridge, y_pred_ridge)
mae_ridge = mean_absolute_error(y_ridge, y_pred_ridge)
rmse_ridge = np.sqrt(mse_ridge)

print(f"=== RESULTADOS MODELO RIDGE ===")
print(f"Alpha utilizado: {best_alpha}")
print(f"R² Score: {r2_ridge:.4f}")
print(f"MSE: {mse_ridge:.4f}")
print(f"RMSE: {rmse_ridge:.4f}")
print(f"MAE: {mae_ridge:.4f}")

print(f"\n=== COMPARACIÓN DE MODELOS ===")
print(f"{'Métrica':<10} {'Reg. Lineal':<12} {'Ridge':<12} {'Diferencia':<12}")
print("-" * 50)
print(f"{'R²':<10} {r2:<12.4f} {r2_ridge:<12.4f} {r2_ridge-r2:<12.4f}")
print(f"{'MSE':<10} {mse:<12.4f} {mse_ridge:<12.4f} {mse_ridge-mse:<12.4f}")
print(f"{'RMSE':<10} {rmse:<12.4f} {rmse_ridge:<12.4f} {rmse_ridge-rmse:<12.4f}")
print(f"{'MAE':<10} {mae:<12.4f} {mae_ridge:<12.4f} {mae_ridge-mae:<12.4f}")

# Mostrar coeficientes de Ridge (en escala original)
print(f"\n=== COEFICIENTES RIDGE ===")
print(f"Intercepto: {modelo_ridge.intercept_:.4f}")
for i, coef in enumerate(modelo_ridge.coef_):
    print(f"{X_ridge.columns[i]}: {coef:.4f}")

# Determinar cuál modelo es mejor
if r2_ridge > r2:
    mejor_modelo = "Ridge"
    print(f"\n🏆 RIDGE es mejor (R² más alto: {r2_ridge:.4f} vs {r2:.4f})")
else:
    mejor_modelo = "Regresión Lineal"
    print(f"\n🏆 REGRESIÓN LINEAL es mejor (R² más alto: {r2:.4f} vs {r2_ridge:.4f})")

=== EVALUACIÓN MODELO RIDGE ===
=== RESULTADOS MODELO RIDGE ===
Alpha utilizado: 10
R² Score: 0.9832
MSE: 1367.0988
RMSE: 36.9743
MAE: 23.8581

=== COMPARACIÓN DE MODELOS ===
Métrica    Reg. Lineal  Ridge        Diferencia  
--------------------------------------------------
R²         0.9883       0.9832       -0.0051     
MSE        954.9393     1367.0988    412.1595    
RMSE       30.9021      36.9743      6.0722      
MAE        21.0201      23.8581      2.8380      

=== COEFICIENTES RIDGE ===
Intercepto: 246.7496
tn: 25.6431
tn_lag_1: 32.5807
tn_lag_2: 23.4058
tn_lag_3: 19.7246
tn_lag_4: 17.9566
tn_lag_5: 20.5549
tn_lag_6: 26.6410
tn_lag_7: 22.7967
tn_lag_8: 24.3096
tn_lag_9: 16.1523
tn_lag_10: 25.0473
tn_lag_11: 29.1412

🏆 REGRESIÓN LINEAL es mejor (R² más alto: 0.9883 vs 0.9832)


In [30]:
# Hacer predicciones para período 202002 usando modelo Ridge
print("=== APLICANDO PREDICCIONES CON RIDGE ===")

# Obtener los coeficientes del modelo Ridge
coef_ridge = modelo_ridge.coef_
intercepto_ridge = modelo_ridge.intercept_

print(f"Intercepto Ridge: {intercepto_ridge:.4f}")
print(f"Coeficientes Ridge: {coef_ridge}")

# Crear dataset para predicciones con Ridge
predicciones_ridge = []

# Obtener todos los product_id únicos (mismo que antes)
productos_unicos = sell_agrup_filtrado_pi['product_id'].unique()

for product_id in productos_unicos:
    # Filtrar datos del producto específico
    datos_producto = sell_agrup_filtrado_pi[sell_agrup_filtrado_pi['product_id'] == product_id].copy()
    datos_producto = datos_producto.sort_values('periodo')
    
    # Verificar si tiene datos para todos los 12 períodos
    periodos_completos = [201901, 201902, 201903, 201904, 201905, 201906,
                         201907, 201908, 201909, 201910, 201911, 201912]
    
    periodos_disponibles = set(datos_producto['periodo'].tolist())
    tiene_todos_periodos = all(p in periodos_disponibles for p in periodos_completos)
    
    if tiene_todos_periodos:
        # CASO 1: Usar el modelo Ridge
        # Obtener valores específicos para la predicción
        tn_201912 = datos_producto[datos_producto['periodo'] == 201912]['tn'].iloc[0]
        tn_201911 = datos_producto[datos_producto['periodo'] == 201911]['tn'].iloc[0]
        tn_201910 = datos_producto[datos_producto['periodo'] == 201910]['tn'].iloc[0]
        tn_201909 = datos_producto[datos_producto['periodo'] == 201909]['tn'].iloc[0]
        tn_201908 = datos_producto[datos_producto['periodo'] == 201908]['tn'].iloc[0]
        tn_201907 = datos_producto[datos_producto['periodo'] == 201907]['tn'].iloc[0]
        tn_201906 = datos_producto[datos_producto['periodo'] == 201906]['tn'].iloc[0]
        tn_201905 = datos_producto[datos_producto['periodo'] == 201905]['tn'].iloc[0]
        tn_201904 = datos_producto[datos_producto['periodo'] == 201904]['tn'].iloc[0]
        tn_201903 = datos_producto[datos_producto['periodo'] == 201903]['tn'].iloc[0]
        tn_201902 = datos_producto[datos_producto['periodo'] == 201902]['tn'].iloc[0]
        tn_201901 = datos_producto[datos_producto['periodo'] == 201901]['tn'].iloc[0]
        
        # Crear array de características (mismo orden que en entrenamiento)
        X_pred = np.array([tn_201912, tn_201911, tn_201910, tn_201909, tn_201908, 
                          tn_201907, tn_201906, tn_201905, tn_201904, tn_201903, 
                          tn_201902, tn_201901]).reshape(1, -1)
        
        # Estandarizar usando el mismo scaler
        X_pred_scaled = scaler.transform(X_pred)
        
        # Hacer predicción con Ridge
        prediccion_ridge = modelo_ridge.predict(X_pred_scaled)[0]
        metodo = "Modelo Ridge"
        
    else:
        # CASO 2: Usar promedio de los períodos disponibles
        promedio = datos_producto['tn'].mean()
        prediccion_ridge = promedio
        metodo = "Promedio Períodos"
    
    # Agregar resultado
    predicciones_ridge.append({
        'product_id': product_id,
        'prediccion_202002_ridge': prediccion_ridge,
        'metodo_usado': metodo,
        'periodos_disponibles': len(datos_producto),
        'tiene_12_periodos': tiene_todos_periodos
    })

# Crear DataFrame con las predicciones Ridge
df_predicciones_ridge = pd.DataFrame(predicciones_ridge)

print(f"\n=== RESUMEN PREDICCIONES RIDGE ===")
print(f"Total productos procesados: {len(df_predicciones_ridge)}")
print(f"Productos con modelo Ridge: {sum(df_predicciones_ridge['metodo_usado'] == 'Modelo Ridge')}")
print(f"Productos con promedio: {sum(df_predicciones_ridge['metodo_usado'] == 'Promedio Períodos')}")

print(f"\n=== PRIMERAS PREDICCIONES RIDGE ===")
print(df_predicciones_ridge.head(10))

=== APLICANDO PREDICCIONES CON RIDGE ===
Intercepto Ridge: 246.7496
Coeficientes Ridge: [25.64309853 32.58069483 23.40577683 19.72461262 17.95661195 20.5548844
 26.64100973 22.79668502 24.3095871  16.15227512 25.04725086 29.14124268]


c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packag


=== RESUMEN PREDICCIONES RIDGE ===
Total productos procesados: 780
Productos con modelo Ridge: 650
Productos con promedio: 130

=== PRIMERAS PREDICCIONES RIDGE ===
   product_id  prediccion_202002_ridge  metodo_usado  periodos_disponibles  \
0       20001              1264.241003  Modelo Ridge                    12   
1       20002              1024.166418  Modelo Ridge                    12   
2       20003               693.731882  Modelo Ridge                    12   
3       20004               538.078480  Modelo Ridge                    12   
4       20005               576.715913  Modelo Ridge                    12   
5       20006               430.574416  Modelo Ridge                    12   
6       20007               380.748705  Modelo Ridge                    12   
7       20008               381.718021  Modelo Ridge                    12   
8       20009               471.246761  Modelo Ridge                    12   
9       20010               380.176562  Modelo Ridge   

c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packag

In [31]:
# Comparar predicciones entre Regresión Lineal y Ridge
print("=== COMPARACIÓN DE PREDICCIONES ===")

# Combinar ambos DataFrames para comparación
df_comparacion = df_predicciones[['product_id', 'prediccion_202002']].merge(
    df_predicciones_ridge[['product_id', 'prediccion_202002_ridge']], 
    on='product_id'
)

# Renombrar columnas para mayor claridad
df_comparacion = df_comparacion.rename(columns={
    'prediccion_202002': 'pred_reg_lineal',
    'prediccion_202002_ridge': 'pred_ridge'
})

# Calcular diferencias
df_comparacion['diferencia'] = df_comparacion['pred_ridge'] - df_comparacion['pred_reg_lineal']
df_comparacion['diferencia_abs'] = abs(df_comparacion['diferencia'])
df_comparacion['diferencia_porcentual'] = (
    df_comparacion['diferencia'] / df_comparacion['pred_reg_lineal'] * 100
).round(2)

print(f"=== ESTADÍSTICAS DE COMPARACIÓN ===")
print(f"Total productos comparados: {len(df_comparacion)}")
print(f"\nRegresión Lineal:")
print(f"  Media: {df_comparacion['pred_reg_lineal'].mean():.2f}")
print(f"  Mediana: {df_comparacion['pred_reg_lineal'].median():.2f}")
print(f"  Min: {df_comparacion['pred_reg_lineal'].min():.2f}")
print(f"  Max: {df_comparacion['pred_reg_lineal'].max():.2f}")

print(f"\nRidge:")
print(f"  Media: {df_comparacion['pred_ridge'].mean():.2f}")
print(f"  Mediana: {df_comparacion['pred_ridge'].median():.2f}")
print(f"  Min: {df_comparacion['pred_ridge'].min():.2f}")
print(f"  Max: {df_comparacion['pred_ridge'].max():.2f}")

print(f"\nDiferencias:")
print(f"  Diferencia media: {df_comparacion['diferencia'].mean():.2f}")
print(f"  Diferencia absoluta media: {df_comparacion['diferencia_abs'].mean():.2f}")
print(f"  Diferencia máxima: {df_comparacion['diferencia_abs'].max():.2f}")
print(f"  Diferencia porcentual media: {df_comparacion['diferencia_porcentual'].mean():.2f}%")

print(f"\n=== PRIMERAS COMPARACIONES ===")
print(df_comparacion.head(10))

# Mostrar productos con mayores diferencias
print(f"\n=== PRODUCTOS CON MAYORES DIFERENCIAS ===")
top_diferencias = df_comparacion.nlargest(5, 'diferencia_abs')
print(top_diferencias[['product_id', 'pred_reg_lineal', 'pred_ridge', 'diferencia', 'diferencia_porcentual']])

=== COMPARACIÓN DE PREDICCIONES ===
=== ESTADÍSTICAS DE COMPARACIÓN ===
Total productos comparados: 780

Regresión Lineal:
  Media: 35.89
  Mediana: 9.04
  Min: 0.05
  Max: 1183.64

Ridge:
  Media: 34.54
  Mediana: 8.36
  Min: -0.28
  Max: 1264.24

Diferencias:
  Diferencia media: -1.36
  Diferencia absoluta media: 2.93
  Diferencia máxima: 159.47
  Diferencia porcentual media: -17.97%

=== PRIMERAS COMPARACIONES ===
   product_id  pred_reg_lineal   pred_ridge  diferencia  diferencia_abs  \
0       20001      1162.707525  1264.241003  101.533478      101.533478   
1       20002      1183.640604  1024.166418 -159.474186      159.474186   
2       20003       684.763931   693.731882    8.967951        8.967951   
3       20004       580.484961   538.078480  -42.406481       42.406481   
4       20005       563.560780   576.715913   13.155132       13.155132   
5       20006       482.886867   430.574416  -52.312450       52.312450   
6       20007       390.924420   380.748705  -10.17571

In [32]:
# Exportar resultados de Ridge y archivo comparativo
print("=== EXPORTANDO RESULTADOS ===")

# 1. Exportar predicciones Ridge (mismo formato que regresión lineal)
df_export_ridge = df_predicciones_ridge[['product_id', 'prediccion_202002_ridge']].copy()
df_export_ridge = df_export_ridge.rename(columns={'prediccion_202002_ridge': 'tn'})

# Definir rutas de salida
drive_base_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression'
filename_ridge = 'predicciones_202002_ridge.csv'
filepath_ridge = os.path.join(drive_base_path, filename_ridge)

filename_comparacion = 'comparacion_modelos.csv'
filepath_comparacion = os.path.join(drive_base_path, filename_comparacion)

# Crear el directorio si no existe
os.makedirs(drive_base_path, exist_ok=True)

# Guardar predicciones Ridge
df_export_ridge.to_csv(filepath_ridge, sep=',', index=False)
print(f"✅ Predicciones Ridge guardadas en: {filepath_ridge}")
print(f"   Total registros: {len(df_export_ridge)}")

# Guardar archivo comparativo
df_comparacion.to_csv(filepath_comparacion, sep=',', index=False)
print(f"✅ Comparación guardada en: {filepath_comparacion}")
print(f"   Total registros: {len(df_comparacion)}")

# 3. Crear archivo con mejor modelo por producto
# Determinar qué modelo usar según métricas de entrenamiento
if r2_ridge > r2:
    modelo_recomendado = "Ridge"
    df_mejor = df_export_ridge.copy()
    print(f"\n🏆 MODELO RECOMENDADO: Ridge (R² = {r2_ridge:.4f})")
else:
    modelo_recomendado = "Regresión Lineal"
    df_mejor = df_export.copy()
    print(f"\n🏆 MODELO RECOMENDADO: Regresión Lineal (R² = {r2:.4f})")

# Guardar predicciones del mejor modelo
filename_mejor = 'predicciones_202002_mejor_modelo.csv'
filepath_mejor = os.path.join(drive_base_path, filename_mejor)
df_mejor.to_csv(filepath_mejor, sep=',', index=False)
print(f"✅ Mejor modelo guardado en: {filepath_mejor}")

# Resumen final
print(f"\n=== RESUMEN FINAL ===")
print(f"📊 Modelos entrenados: Regresión Lineal + Ridge")
print(f"📈 Mejor modelo: {modelo_recomendado}")
print(f"📁 Archivos generados:")
print(f"   - Regresión Lineal: predicciones_202002.csv")
print(f"   - Ridge: predicciones_202002_ridge.csv") 
print(f"   - Comparación: comparacion_modelos.csv")
print(f"   - Mejor modelo: predicciones_202002_mejor_modelo.csv")

# Verificar archivos
for filepath, nombre in [(filepath_ridge, "Ridge"), (filepath_comparacion, "Comparación"), (filepath_mejor, "Mejor modelo")]:
    if os.path.exists(filepath):
        file_size = os.path.getsize(filepath)
        print(f"✅ {nombre}: {file_size} bytes")
    else:
        print(f"❌ {nombre}: Error al crear archivo")

=== EXPORTANDO RESULTADOS ===
✅ Predicciones Ridge guardadas en: C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression\predicciones_202002_ridge.csv
   Total registros: 780
✅ Comparación guardada en: C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression\comparacion_modelos.csv
   Total registros: 780

🏆 MODELO RECOMENDADO: Regresión Lineal (R² = 0.9883)
✅ Mejor modelo guardado en: C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression\predicciones_202002_mejor_modelo.csv

=== RESUMEN FINAL ===
📊 Modelos entrenados: Regresión Lineal + Ridge
📈 Mejor modelo: Regresión Lineal
📁 Archivos generados:
   - Regresión Lineal: predicciones_202002.csv
   - Ridge: predicciones_202002_ridge.csv
   - Comparación: comparacion_modelos.csv
   - Mejor modelo: predicciones_202002_mejor_modelo.csv
✅ Ridge: 19573 bytes
✅ Comparación: 64263 bytes
✅ Mejor modelo: 19470 bytes


## Predicción con modelo regresión Lasso

In [33]:
# Importar Lasso y herramientas adicionales para comparación de modelos
from sklearn.linear_model import Lasso

print("=== CONFIGURACIÓN MODELO LASSO ===")

# Preparar los mismos datos que usamos para regresión lineal y Ridge
X_lasso = X_clean.copy()
y_lasso = y_clean.copy()

print(f"Datos para Lasso - X shape: {X_lasso.shape}, y shape: {y_lasso.shape}")

# Usar el mismo scaler que Ridge (importante para comparación)
X_lasso_scaled = scaler.transform(X_lasso)

print("Variables estandarizadas para modelo Lasso (usando mismo scaler que Ridge)")
print(f"Media de X_lasso_scaled: {X_lasso_scaled.mean(axis=0)}")
print(f"Std de X_lasso_scaled: {X_lasso_scaled.std(axis=0)}")

=== CONFIGURACIÓN MODELO LASSO ===
Datos para Lasso - X shape: (33, 12), y shape: (33,)
Variables estandarizadas para modelo Lasso (usando mismo scaler que Ridge)
Media de X_lasso_scaled: [ 9.42007415e-17 -1.68215610e-17  9.08364293e-17  2.69144976e-17
  1.78308546e-16 -1.44665424e-16 -1.07657990e-16 -4.37360585e-17
  7.73791805e-17 -1.00929366e-17  4.37360585e-17 -1.00929366e-16]
Std de X_lasso_scaled: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [34]:
# Buscar el mejor valor de alpha (regularización) para Lasso
print("=== BÚSQUEDA DEL MEJOR ALPHA PARA LASSO ===")

# Definir rango de valores alpha para probar (similar a Ridge pero ajustado para Lasso)
alphas_lasso = [0.001, 0.01, 0.1, 1, 10, 100]
best_alpha_lasso = None
best_score_lasso = -np.inf
alpha_scores_lasso = []

# Probar cada valor de alpha con validación cruzada
for alpha in alphas_lasso:
    lasso_temp = Lasso(alpha=alpha, max_iter=2000)  # Aumentar max_iter para convergencia
    # Usar validación cruzada (si hay suficientes datos)
    if len(X_lasso_scaled) >= 5:  # Mínimo 5 muestras para CV
        scores = cross_val_score(lasso_temp, X_lasso_scaled, y_lasso, cv=min(5, len(X_lasso_scaled)), scoring='r2')
        mean_score = scores.mean()
    else:
        # Si hay pocos datos, usar el R² del ajuste completo
        lasso_temp.fit(X_lasso_scaled, y_lasso)
        y_pred_temp = lasso_temp.predict(X_lasso_scaled)
        mean_score = r2_score(y_lasso, y_pred_temp)
    
    alpha_scores_lasso.append(mean_score)
    print(f"Alpha: {alpha:>6} - R² promedio: {mean_score:.4f}")
    
    if mean_score > best_score_lasso:
        best_score_lasso = mean_score
        best_alpha_lasso = alpha

print(f"\n=== MEJOR CONFIGURACIÓN LASSO ===")
print(f"Mejor alpha: {best_alpha_lasso}")
print(f"Mejor R² score: {best_score_lasso:.4f}")

# Entrenar el modelo Lasso final con el mejor alpha
modelo_lasso = Lasso(alpha=best_alpha_lasso, max_iter=2000)
modelo_lasso.fit(X_lasso_scaled, y_lasso)

# Verificar convergencia
if modelo_lasso.n_iter_ == modelo_lasso.max_iter:
    print("⚠️  ADVERTENCIA: Lasso no convergió completamente, considerando aumentar max_iter")
else:
    print(f"✅ Lasso convergió en {modelo_lasso.n_iter_} iteraciones")

=== BÚSQUEDA DEL MEJOR ALPHA PARA LASSO ===
Alpha:  0.001 - R² promedio: -1.1775
Alpha:   0.01 - R² promedio: -1.0814
Alpha:    0.1 - R² promedio: -0.4367
Alpha:      1 - R² promedio: -0.1779
Alpha:     10 - R² promedio: -1.1452
Alpha:    100 - R² promedio: -135.5418

=== MEJOR CONFIGURACIÓN LASSO ===
Mejor alpha: 1
Mejor R² score: -0.1779
✅ Lasso convergió en 361 iteraciones


c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.776e+03, tolerance: 2.668e+02
  model = cd_fast.enet_coordinate_descent(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.256e+03, tolerance: 2.600e+02
  model = cd_fast.enet_coordinate_descent(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Du

In [35]:
# Evaluar el modelo Lasso y comparar con regresión lineal y Ridge
print("=== EVALUACIÓN MODELO LASSO ===")

# Hacer predicciones con Lasso
y_pred_lasso = modelo_lasso.predict(X_lasso_scaled)

# Calcular métricas para Lasso
r2_lasso = r2_score(y_lasso, y_pred_lasso)
mse_lasso = mean_squared_error(y_lasso, y_pred_lasso)
mae_lasso = mean_absolute_error(y_lasso, y_pred_lasso)
rmse_lasso = np.sqrt(mse_lasso)

print(f"=== RESULTADOS MODELO LASSO ===")
print(f"Alpha utilizado: {best_alpha_lasso}")
print(f"R² Score: {r2_lasso:.4f}")
print(f"MSE: {mse_lasso:.4f}")
print(f"RMSE: {rmse_lasso:.4f}")
print(f"MAE: {mae_lasso:.4f}")

print(f"\n=== COMPARACIÓN DE LOS 3 MODELOS ===")
print(f"{'Métrica':<10} {'Reg. Lineal':<12} {'Ridge':<12} {'Lasso':<12}")
print("-" * 55)
print(f"{'R²':<10} {r2:<12.4f} {r2_ridge:<12.4f} {r2_lasso:<12.4f}")
print(f"{'MSE':<10} {mse:<12.4f} {mse_ridge:<12.4f} {mse_lasso:<12.4f}")
print(f"{'RMSE':<10} {rmse:<12.4f} {rmse_ridge:<12.4f} {rmse_lasso:<12.4f}")
print(f"{'MAE':<10} {mae:<12.4f} {mae_ridge:<12.4f} {mae_lasso:<12.4f}")

# Mostrar coeficientes de Lasso y cuáles fueron eliminados (=0)
print(f"\n=== COEFICIENTES LASSO ===")
print(f"Intercepto: {modelo_lasso.intercept_:.4f}")
coeficientes_no_cero = 0
for i, coef in enumerate(modelo_lasso.coef_):
    if abs(coef) > 1e-10:  # Considerar prácticamente cero
        print(f"{X_lasso.columns[i]}: {coef:.4f}")
        coeficientes_no_cero += 1
    else:
        print(f"{X_lasso.columns[i]}: 0.0000 (eliminado)")

print(f"\nVariables seleccionadas por Lasso: {coeficientes_no_cero}/{len(modelo_lasso.coef_)}")

# Determinar cuál modelo es mejor basado en R²
modelos_r2 = {
    "Regresión Lineal": r2,
    "Ridge": r2_ridge, 
    "Lasso": r2_lasso
}

mejor_modelo_global = max(modelos_r2, key=modelos_r2.get)
mejor_r2_global = modelos_r2[mejor_modelo_global]

print(f"\n🏆 MEJOR MODELO GLOBAL: {mejor_modelo_global} (R² = {mejor_r2_global:.4f})")

# Comparación específica con Ridge
if r2_lasso > r2_ridge:
    print(f"📈 Lasso supera a Ridge por {r2_lasso - r2_ridge:.4f} puntos de R²")
elif r2_ridge > r2_lasso:
    print(f"📈 Ridge supera a Lasso por {r2_ridge - r2_lasso:.4f} puntos de R²")
else:
    print(f"⚖️  Lasso y Ridge tienen rendimiento similar")

=== EVALUACIÓN MODELO LASSO ===
=== RESULTADOS MODELO LASSO ===
Alpha utilizado: 1
R² Score: 0.9874
MSE: 1029.2474
RMSE: 32.0819
MAE: 22.3145

=== COMPARACIÓN DE LOS 3 MODELOS ===
Métrica    Reg. Lineal  Ridge        Lasso       
-------------------------------------------------------
R²         0.9883       0.9832       0.9874      
MSE        954.9393     1367.0988    1029.2474   
RMSE       30.9021      36.9743      32.0819     
MAE        21.0201      23.8581      22.3145     

=== COEFICIENTES LASSO ===
Intercepto: 246.7496
tn: 7.2840
tn_lag_1: 84.8357
tn_lag_2: 38.2357
tn_lag_3: 0.0000 (eliminado)
tn_lag_4: 0.0000 (eliminado)
tn_lag_5: 0.0000 (eliminado)
tn_lag_6: 39.3534
tn_lag_7: 0.0000 (eliminado)
tn_lag_8: 45.7773
tn_lag_9: 15.3007
tn_lag_10: 33.6315
tn_lag_11: 28.2154

Variables seleccionadas por Lasso: 8/12

🏆 MEJOR MODELO GLOBAL: Regresión Lineal (R² = 0.9883)
📈 Lasso supera a Ridge por 0.0041 puntos de R²


In [36]:
# Hacer predicciones para período 202002 usando modelo Lasso
print("=== APLICANDO PREDICCIONES CON LASSO ===")

# Obtener los coeficientes del modelo Lasso
coef_lasso = modelo_lasso.coef_
intercepto_lasso = modelo_lasso.intercept_

print(f"Intercepto Lasso: {intercepto_lasso:.4f}")
print(f"Coeficientes Lasso: {coef_lasso}")
print(f"Variables seleccionadas: {sum(abs(coef) > 1e-10 for coef in coef_lasso)}/{len(coef_lasso)}")

# Crear dataset para predicciones con Lasso
predicciones_lasso = []

# Obtener todos los product_id únicos (mismo que antes)
productos_unicos = sell_agrup_filtrado_pi['product_id'].unique()

for product_id in productos_unicos:
    # Filtrar datos del producto específico
    datos_producto = sell_agrup_filtrado_pi[sell_agrup_filtrado_pi['product_id'] == product_id].copy()
    datos_producto = datos_producto.sort_values('periodo')
    
    # Verificar si tiene datos para todos los 12 períodos
    periodos_completos = [201901, 201902, 201903, 201904, 201905, 201906,
                         201907, 201908, 201909, 201910, 201911, 201912]
    
    periodos_disponibles = set(datos_producto['periodo'].tolist())
    tiene_todos_periodos = all(p in periodos_disponibles for p in periodos_completos)
    
    if tiene_todos_periodos:
        # CASO 1: Usar el modelo Lasso
        # Obtener valores específicos para la predicción
        tn_201912 = datos_producto[datos_producto['periodo'] == 201912]['tn'].iloc[0]
        tn_201911 = datos_producto[datos_producto['periodo'] == 201911]['tn'].iloc[0]
        tn_201910 = datos_producto[datos_producto['periodo'] == 201910]['tn'].iloc[0]
        tn_201909 = datos_producto[datos_producto['periodo'] == 201909]['tn'].iloc[0]
        tn_201908 = datos_producto[datos_producto['periodo'] == 201908]['tn'].iloc[0]
        tn_201907 = datos_producto[datos_producto['periodo'] == 201907]['tn'].iloc[0]
        tn_201906 = datos_producto[datos_producto['periodo'] == 201906]['tn'].iloc[0]
        tn_201905 = datos_producto[datos_producto['periodo'] == 201905]['tn'].iloc[0]
        tn_201904 = datos_producto[datos_producto['periodo'] == 201904]['tn'].iloc[0]
        tn_201903 = datos_producto[datos_producto['periodo'] == 201903]['tn'].iloc[0]
        tn_201902 = datos_producto[datos_producto['periodo'] == 201902]['tn'].iloc[0]
        tn_201901 = datos_producto[datos_producto['periodo'] == 201901]['tn'].iloc[0]
        
        # Crear array de características (mismo orden que en entrenamiento)
        X_pred = np.array([tn_201912, tn_201911, tn_201910, tn_201909, tn_201908, 
                          tn_201907, tn_201906, tn_201905, tn_201904, tn_201903, 
                          tn_201902, tn_201901]).reshape(1, -1)
        
        # Estandarizar usando el mismo scaler
        X_pred_scaled = scaler.transform(X_pred)
        
        # Hacer predicción con Lasso
        prediccion_lasso = modelo_lasso.predict(X_pred_scaled)[0]
        metodo = "Modelo Lasso"
        
    else:
        # CASO 2: Usar promedio de los períodos disponibles
        promedio = datos_producto['tn'].mean()
        prediccion_lasso = promedio
        metodo = "Promedio Períodos"
    
    # Agregar resultado
    predicciones_lasso.append({
        'product_id': product_id,
        'prediccion_202002_lasso': prediccion_lasso,
        'metodo_usado': metodo,
        'periodos_disponibles': len(datos_producto),
        'tiene_12_periodos': tiene_todos_periodos
    })

# Crear DataFrame con las predicciones Lasso
df_predicciones_lasso = pd.DataFrame(predicciones_lasso)

print(f"\n=== RESUMEN PREDICCIONES LASSO ===")
print(f"Total productos procesados: {len(df_predicciones_lasso)}")
print(f"Productos con modelo Lasso: {sum(df_predicciones_lasso['metodo_usado'] == 'Modelo Lasso')}")
print(f"Productos con promedio: {sum(df_predicciones_lasso['metodo_usado'] == 'Promedio Períodos')}")

print(f"\n=== PRIMERAS PREDICCIONES LASSO ===")
print(df_predicciones_lasso.head(10))

=== APLICANDO PREDICCIONES CON LASSO ===
Intercepto Lasso: 246.7496
Coeficientes Lasso: [ 7.2839855  84.83571838 38.23574886  0.          0.          0.
 39.35339202  0.         45.77729157 15.30066024 33.63149523 28.21536856]
Variables seleccionadas: 8/12


c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packag


=== RESUMEN PREDICCIONES LASSO ===
Total productos procesados: 780
Productos con modelo Lasso: 650
Productos con promedio: 130

=== PRIMERAS PREDICCIONES LASSO ===
   product_id  prediccion_202002_lasso  metodo_usado  periodos_disponibles  \
0       20001              1190.054107  Modelo Lasso                    12   
1       20002              1094.308084  Modelo Lasso                    12   
2       20003               692.947197  Modelo Lasso                    12   
3       20004               541.181072  Modelo Lasso                    12   
4       20005               541.943462  Modelo Lasso                    12   
5       20006               460.928722  Modelo Lasso                    12   
6       20007               377.172897  Modelo Lasso                    12   
7       20008               394.700348  Modelo Lasso                    12   
8       20009               456.508591  Modelo Lasso                    12   
9       20010               391.460390  Modelo Lasso   

c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packag

In [37]:
# Comparar predicciones entre los 3 modelos: Regresión Lineal, Ridge y Lasso
print("=== COMPARACIÓN DE PREDICCIONES DE LOS 3 MODELOS ===")

# Combinar los 3 DataFrames para comparación completa
df_comparacion_completa = (df_predicciones[['product_id', 'prediccion_202002']]
                          .merge(df_predicciones_ridge[['product_id', 'prediccion_202002_ridge']], on='product_id')
                          .merge(df_predicciones_lasso[['product_id', 'prediccion_202002_lasso']], on='product_id'))

# Renombrar columnas para mayor claridad
df_comparacion_completa = df_comparacion_completa.rename(columns={
    'prediccion_202002': 'pred_reg_lineal',
    'prediccion_202002_ridge': 'pred_ridge',
    'prediccion_202002_lasso': 'pred_lasso'
})

# Calcular diferencias entre modelos
df_comparacion_completa['dif_ridge_lineal'] = df_comparacion_completa['pred_ridge'] - df_comparacion_completa['pred_reg_lineal']
df_comparacion_completa['dif_lasso_lineal'] = df_comparacion_completa['pred_lasso'] - df_comparacion_completa['pred_reg_lineal']
df_comparacion_completa['dif_lasso_ridge'] = df_comparacion_completa['pred_lasso'] - df_comparacion_completa['pred_ridge']

# Calcular diferencias absolutas
df_comparacion_completa['dif_abs_ridge_lineal'] = abs(df_comparacion_completa['dif_ridge_lineal'])
df_comparacion_completa['dif_abs_lasso_lineal'] = abs(df_comparacion_completa['dif_lasso_lineal'])
df_comparacion_completa['dif_abs_lasso_ridge'] = abs(df_comparacion_completa['dif_lasso_ridge'])

print(f"=== ESTADÍSTICAS COMPARATIVAS DE LOS 3 MODELOS ===")
print(f"Total productos comparados: {len(df_comparacion_completa)}")

print(f"\n📊 ESTADÍSTICAS DESCRIPTIVAS:")
for modelo, columna in [("Regresión Lineal", "pred_reg_lineal"), ("Ridge", "pred_ridge"), ("Lasso", "pred_lasso")]:
    datos = df_comparacion_completa[columna]
    print(f"\n{modelo}:")
    print(f"  Media: {datos.mean():.2f}")
    print(f"  Mediana: {datos.median():.2f}")
    print(f"  Desv. Std: {datos.std():.2f}")
    print(f"  Min: {datos.min():.2f}")
    print(f"  Max: {datos.max():.2f}")

print(f"\n📈 DIFERENCIAS PROMEDIO ENTRE MODELOS:")
print(f"Ridge vs Reg. Lineal: {df_comparacion_completa['dif_ridge_lineal'].mean():.2f} (absoluta: {df_comparacion_completa['dif_abs_ridge_lineal'].mean():.2f})")
print(f"Lasso vs Reg. Lineal: {df_comparacion_completa['dif_lasso_lineal'].mean():.2f} (absoluta: {df_comparacion_completa['dif_abs_lasso_lineal'].mean():.2f})")
print(f"Lasso vs Ridge: {df_comparacion_completa['dif_lasso_ridge'].mean():.2f} (absoluta: {df_comparacion_completa['dif_abs_lasso_ridge'].mean():.2f})")

print(f"\n=== PRIMERAS COMPARACIONES DE LOS 3 MODELOS ===")
print(df_comparacion_completa[['product_id', 'pred_reg_lineal', 'pred_ridge', 'pred_lasso', 
                               'dif_ridge_lineal', 'dif_lasso_lineal', 'dif_lasso_ridge']].head(10))

# Mostrar productos con mayores diferencias entre Lasso y Ridge
print(f"\n=== PRODUCTOS CON MAYORES DIFERENCIAS LASSO vs RIDGE ===")
top_diferencias_lasso_ridge = df_comparacion_completa.nlargest(5, 'dif_abs_lasso_ridge')
print(top_diferencias_lasso_ridge[['product_id', 'pred_ridge', 'pred_lasso', 'dif_lasso_ridge']])

# Encontrar productos donde los 3 modelos coinciden más
df_comparacion_completa['varianza_predicciones'] = df_comparacion_completa[['pred_reg_lineal', 'pred_ridge', 'pred_lasso']].var(axis=1)
print(f"\n=== PRODUCTOS CON MAYOR CONSENSO (menor varianza entre modelos) ===")
top_consenso = df_comparacion_completa.nsmallest(5, 'varianza_predicciones')
print(top_consenso[['product_id', 'pred_reg_lineal', 'pred_ridge', 'pred_lasso', 'varianza_predicciones']])

=== COMPARACIÓN DE PREDICCIONES DE LOS 3 MODELOS ===
=== ESTADÍSTICAS COMPARATIVAS DE LOS 3 MODELOS ===
Total productos comparados: 780

📊 ESTADÍSTICAS DESCRIPTIVAS:

Regresión Lineal:
  Media: 35.89
  Mediana: 9.04
  Desv. Std: 92.87
  Min: 0.05
  Max: 1183.64

Ridge:
  Media: 34.54
  Mediana: 8.36
  Desv. Std: 90.73
  Min: -0.28
  Max: 1264.24

Lasso:
  Media: 33.76
  Mediana: 7.53
  Desv. Std: 90.72
  Min: -1.43
  Max: 1190.05

📈 DIFERENCIAS PROMEDIO ENTRE MODELOS:
Ridge vs Reg. Lineal: -1.36 (absoluta: 2.93)
Lasso vs Reg. Lineal: -2.14 (absoluta: 2.65)
Lasso vs Ridge: -0.78 (absoluta: 2.03)

=== PRIMERAS COMPARACIONES DE LOS 3 MODELOS ===
   product_id  pred_reg_lineal   pred_ridge   pred_lasso  dif_ridge_lineal  \
0       20001      1162.707525  1264.241003  1190.054107        101.533478   
1       20002      1183.640604  1024.166418  1094.308084       -159.474186   
2       20003       684.763931   693.731882   692.947197          8.967951   
3       20004       580.484961   538.

In [38]:
# Exportar resultados de Lasso y archivos comparativos finales
print("=== EXPORTANDO RESULTADOS FINALES ===")

# 1. Exportar predicciones Lasso (mismo formato que los otros modelos)
df_export_lasso = df_predicciones_lasso[['product_id', 'prediccion_202002_lasso']].copy()
df_export_lasso = df_export_lasso.rename(columns={'prediccion_202002_lasso': 'tn'})

# Definir rutas de salida
drive_base_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression'
filename_lasso = 'predicciones_202002_lasso.csv'
filepath_lasso = os.path.join(drive_base_path, filename_lasso)

filename_comparacion_completa = 'comparacion_3_modelos.csv'
filepath_comparacion_completa = os.path.join(drive_base_path, filename_comparacion_completa)

# Crear el directorio si no existe
os.makedirs(drive_base_path, exist_ok=True)

# Guardar predicciones Lasso
df_export_lasso.to_csv(filepath_lasso, sep=',', index=False)
print(f"✅ Predicciones Lasso guardadas en: {filepath_lasso}")
print(f"   Total registros: {len(df_export_lasso)}")

# Guardar archivo comparativo completo de los 3 modelos
df_comparacion_completa.to_csv(filepath_comparacion_completa, sep=',', index=False)
print(f"✅ Comparación completa guardada en: {filepath_comparacion_completa}")
print(f"   Total registros: {len(df_comparacion_completa)}")

# 3. Determinar el mejor modelo global y crear archivo final
modelos_performance = {
    "Regresión Lineal": r2,
    "Ridge": r2_ridge,
    "Lasso": r2_lasso
}

mejor_modelo_final = max(modelos_performance, key=modelos_performance.get)
mejor_r2_final = modelos_performance[mejor_modelo_final]

# Seleccionar el DataFrame del mejor modelo
if mejor_modelo_final == "Regresión Lineal":
    df_mejor_final = df_export.copy()
elif mejor_modelo_final == "Ridge":
    df_mejor_final = df_export_ridge.copy()
else:  # Lasso
    df_mejor_final = df_export_lasso.copy()

# Guardar predicciones del mejor modelo (actualizada)
filename_mejor_final = 'predicciones_202002_mejor_modelo_final.csv'
filepath_mejor_final = os.path.join(drive_base_path, filename_mejor_final)
df_mejor_final.to_csv(filepath_mejor_final, sep=',', index=False)
print(f"✅ Mejor modelo final guardado en: {filepath_mejor_final}")

# 4. Crear archivo con promedio de los 3 modelos (ensemble)
df_ensemble = df_comparacion_completa[['product_id']].copy()
df_ensemble['tn'] = (df_comparacion_completa['pred_reg_lineal'] + 
                     df_comparacion_completa['pred_ridge'] + 
                     df_comparacion_completa['pred_lasso']) / 3

filename_ensemble = 'predicciones_202002_ensemble.csv'
filepath_ensemble = os.path.join(drive_base_path, filename_ensemble)
df_ensemble.to_csv(filepath_ensemble, sep=',', index=False)
print(f"✅ Predicciones Ensemble guardadas en: {filepath_ensemble}")

# Resumen final completo
print(f"\n=== RESUMEN FINAL COMPLETO ===")
print(f"📊 Modelos entrenados: Regresión Lineal + Ridge + Lasso")
print(f"📈 Ranking de modelos por R²:")
for i, (modelo, r2_val) in enumerate(sorted(modelos_performance.items(), key=lambda x: x[1], reverse=True), 1):
    print(f"   {i}. {modelo}: {r2_val:.4f}")

print(f"\n🏆 MEJOR MODELO: {mejor_modelo_final} (R² = {mejor_r2_final:.4f})")

print(f"\n📁 Archivos generados:")
print(f"   - Regresión Lineal: predicciones_202002.csv")
print(f"   - Ridge: predicciones_202002_ridge.csv")
print(f"   - Lasso: predicciones_202002_lasso.csv")
print(f"   - Comparación 2 modelos: comparacion_modelos.csv")
print(f"   - Comparación 3 modelos: comparacion_3_modelos.csv")
print(f"   - Mejor modelo: predicciones_202002_mejor_modelo_final.csv")
print(f"   - Ensemble (promedio): predicciones_202002_ensemble.csv")

# Verificar todos los archivos
archivos_verificar = [
    (filepath_lasso, "Lasso"),
    (filepath_comparacion_completa, "Comparación 3 modelos"),
    (filepath_mejor_final, "Mejor modelo final"),
    (filepath_ensemble, "Ensemble")
]

print(f"\n📋 Verificación de archivos:")
for filepath, nombre in archivos_verificar:
    if os.path.exists(filepath):
        file_size = os.path.getsize(filepath)
        print(f"✅ {nombre}: {file_size} bytes")
    else:
        print(f"❌ {nombre}: Error al crear archivo")

print(f"\n🎯 RECOMENDACIÓN: Usar el archivo '{filename_mejor_final}' para las predicciones finales")
print(f"   Alternativamente, '{filename_ensemble}' combina los 3 modelos para mayor robustez")

=== EXPORTANDO RESULTADOS FINALES ===
✅ Predicciones Lasso guardadas en: C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression\predicciones_202002_lasso.csv
   Total registros: 780
✅ Comparación completa guardada en: C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression\comparacion_3_modelos.csv
   Total registros: 780
✅ Mejor modelo final guardado en: C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression\predicciones_202002_mejor_modelo_final.csv
✅ Predicciones Ensemble guardadas en: C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression\predicciones_202002_ensemble.csv

=== RESUMEN FINAL COMPLETO ===
📊 Modelos entrenados: Regresión Lineal + Ridge + Lasso
📈 Ranking de modelos por R²:
   1. Regresión Lineal: 0.9883
   2. Lasso: 0.9874
   3. Ridge: 0.9832

🏆 MEJOR MODELO: Regresión Lineal (R² = 0.9883)

📁 Archivos generados:
   - Regresión Lineal: predicciones_202002.csv
   - Ridge: predicciones_202002_ridge.csv
